In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
CLEANED_PATH = "../data/processed/cleaned_data.csv"

METADATA_PATH = "../data/processed/graph_metadata.csv"

EDGES_PATH = "../data/processed/graph_edges.csv"

NODES_PATH = "../data/processed/graph_nodes.csv"

OUTPUT_FEATURES = "../data/processed/node_features.csv"

OUTPUT_LABELS = "../data/processed/node_labels.csv"

OUTPUT_NODES = "../data/processed/final_nodes.csv"

In [3]:
input_files = [
    CLEANED_PATH,
    METADATA_PATH,
    EDGES_PATH,
    NODES_PATH
]

for file_path in input_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found:\n{file_path}"
        )

print("All input files found.")

All input files found.


In [4]:
cleaned_df = pd.read_csv(CLEANED_PATH)

metadata_df = pd.read_csv(METADATA_PATH)

edges_df = pd.read_csv(EDGES_PATH)

nodes_df = pd.read_csv(NODES_PATH)

print("Cleaned data:", cleaned_df.shape)

print("Metadata:", metadata_df.shape)

print("Edges:", edges_df.shape)

print("Nodes:", nodes_df.shape)

Cleaned data: (286095, 68)
Metadata: (286095, 9)
Edges: (50000, 9)
Nodes: (3001, 9)


In [5]:
if len(cleaned_df) != len(metadata_df):
    raise ValueError(
        "Cleaned data and metadata row counts do not match."
    )

print("Row counts match.")

Row counts match.


In [6]:
graph_df = metadata_df.copy()

feature_columns = [
    column
    for column in cleaned_df.columns
    if column != "Binary_Label"
]

for column in feature_columns:
    graph_df[column] = cleaned_df[column].values

print("Combined dataset shape:", graph_df.shape)

Combined dataset shape: (286095, 76)


In [7]:
print(graph_df.columns.tolist())

['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Label', 'Binary_Label', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Cou

In [8]:
excluded_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label",
    "Binary_Label"
]

traffic_features = [
    column
    for column in graph_df.columns
    if column not in excluded_columns
]

print("Number of traffic features:", len(traffic_features))

print(traffic_features)

Number of traffic features: 67
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segmen

In [9]:
for column in traffic_features:
    graph_df[column] = pd.to_numeric(
        graph_df[column],
        errors="coerce"
    )

graph_df[traffic_features] = (
    graph_df[traffic_features]
    .replace([np.inf, -np.inf], np.nan)
)

graph_df = graph_df.dropna(
    subset=traffic_features
).reset_index(drop=True)

print("Rows after cleaning:", len(graph_df))

Rows after cleaning: 286095


In [10]:
source_features = (
    graph_df
    .groupby("Source IP")[traffic_features]
    .mean()
)

source_features.columns = [
    f"source_{column}"
    for column in source_features.columns
]

source_features.head()

,source_Flow Duration,source_Total Fwd Packets,source_Total Backward Packets,source_Total Length of Fwd Packets,source_Total Length of Bwd Packets,source_Fwd Packet Length Max,source_Fwd Packet Length Min,source_Fwd Packet Length Mean,source_Fwd Packet Length Std,source_Bwd Packet Length Max,source_Bwd Packet Length Min,source_Bwd Packet Length Mean,source_Bwd Packet Length Std,source_Flow Bytes/s,source_Flow Packets/s,source_Flow IAT Mean,source_Flow IAT Std,source_Flow IAT Max,source_Flow IAT Min,source_Fwd IAT Total,source_Fwd IAT Mean,source_Fwd IAT Std,source_Fwd IAT Max,source_Fwd IAT Min,source_Bwd IAT Total,source_Bwd IAT Mean,source_Bwd IAT Std,source_Bwd IAT Max,source_Bwd IAT Min,source_Fwd PSH Flags,source_Fwd Header Length,source_Bwd Header Length,source_Fwd Packets/s,source_Bwd Packets/s,source_Min Packet Length,source_Max Packet Length,source_Packet Length Mean,source_Packet Length Std,source_Packet Length Variance,source_FIN Flag Count,source_SYN Flag Count,source_RST Flag Count,source_PSH Flag Count,source_ACK Flag Count,source_URG Flag Count,source_ECE Flag Count,source_Down/Up Ratio,source_Average Packet Size,source_Avg Fwd Segment Size,source_Avg Bwd Segment Size,source_Fwd Header Length.1,source_Subflow Fwd Packets,source_Subflow Fwd Bytes,source_Subflow Bwd Packets,source_Subflow Bwd Bytes,source_Init_Win_bytes_forward,source_Init_Win_bytes_backward,source_act_data_pkt_fwd,source_min_seg_size_forward,source_Active Mean,source_Active Std,source_Active Max,source_Active Min,source_Idle Mean,source_Idle Std,source_Idle Max,source_Idle Min
Source IP,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
101.37.178.168,-0.233453,-0.075601,-0.089164,-0.118767,-0.053023,-0.229084,-0.179794,-0.231604,-0.209924,-0.306943,-0.396604,-0.345473,-0.283668,-0.043512,-0.324571,-0.099454,-0.125884,-0.172413,-0.030272,-0.229113,-0.086079,-0.197619,-0.167388,-0.031810,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,-0.146348,-0.103088,-0.096237,-0.258753,-0.338118,-0.220255,-0.315617,-0.343325,-0.346768,-0.218248,-0.0903,-0.146348,-0.008568,-1.396722,2.664771,4.622056,-0.008568,-1.888918,-0.363615,-0.231604,-0.345473,-0.103088,-0.075601,-0.118767,-0.089164,-0.053023,-0.779718,-0.134468,-0.049019,-1.133138,-0.073218,-0.087551,-0.106105,-0.05145,-0.184749,-0.050323,-0.185562,-0.179561
103.203.138.103,-0.245533,-0.075601,-0.124491,-0.118767,-0.053141,-0.229084,-0.179794,-0.231604,-0.209924,-0.317180,-0.507229,-0.374846,-0.283668,0.064012,2.071561,-0.161755,-0.174779,-0.201726,-0.030156,-0.241264,-0.130372,-0.197619,-0.196752,-0.077659,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,-0.146348,-0.103088,-0.124862,2.614553,-0.338178,-0.220255,-0.315617,-0.343325,-0.346768,-0.218248,-0.0903,-0.146348,-0.008568,-1.396722,2.664771,-0.216354,-0.008568,-1.888918,-0.355915,-0.231604,-0.374846,-0.103088,-0.075601,-0.118767,-0.124491,-0.053141,-0.745321,-0.173093,-0.049019,-1.133138,-0.073218,-0.087551,-0.106105,-0.05145,-0.184749,-0.050323,-0.185562,-0.179561
103.23.4.38,-0.245536,-0.075601,-0.124491,-0.108589,-0.053141,-0.152858,-0.427194,-0.110475,-0.023036,-0.317180,-0.507229,-0.374846,-0.283668,1.163504,10.087452,-0.161786,-0.174779,-0.201734,-0.030330,-0.241267,-0.130383,-0.197619,-0.196759,-0.077670,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,6.833041,-0.055457,-0.124862,12.226673,-0.338178,-0.539775,-0.276370,-0.223205,-0.246394,-0.216285,-0.0903,6.833041,-0.008568,-1.396722,2.664771,-0.216354,-0.008568,-1.888918,-0.186510,-0.110475,-0.374846,-0.055457,-0.075601,-0.108589,-0.124491,-0.053141,-0.772572,-0.173093,-0.116102,0.364318,-0.073218,-0.087551,-0.106105,-0.05145,-0.184749,-0.050323,-0.185562,-0.179561
103.235.247.8,-0.202545,-0.106327,-0.089164,-0.092946,-0.051828,-0.067484,-0.122067,0.115207,0.101155,-0.206953,0.683832,-0.058594,-0.283668,0.471911,0.212010,0.281724,-0.174776,-0.097411,2.449414,-0.240060,-0.125985,-0.197619,-0.193844,-0.073118,-0.233928,-0.127790,-0.167158,-0.180280,-0.083686,2.645408,

In [11]:
destination_features = (
    graph_df
    .groupby("Destination IP")[traffic_features]
    .mean()
)

destination_features.columns = [
    f"destination_{column}"
    for column in destination_features.columns
]

destination_features.head()

,destination_Flow Duration,destination_Total Fwd Packets,destination_Total Backward Packets,destination_Total Length of Fwd Packets,destination_Total Length of Bwd Packets,destination_Fwd Packet Length Max,destination_Fwd Packet Length Min,destination_Fwd Packet Length Mean,destination_Fwd Packet Length Std,destination_Bwd Packet Length Max,destination_Bwd Packet Length Min,destination_Bwd Packet Length Mean,destination_Bwd Packet Length Std,destination_Flow Bytes/s,destination_Flow Packets/s,destination_Flow IAT Mean,destination_Flow IAT Std,destination_Flow IAT Max,destination_Flow IAT Min,destination_Fwd IAT Total,destination_Fwd IAT Mean,destination_Fwd IAT Std,destination_Fwd IAT Max,destination_Fwd IAT Min,destination_Bwd IAT Total,destination_Bwd IAT Mean,destination_Bwd IAT Std,destination_Bwd IAT Max,destination_Bwd IAT Min,destination_Fwd PSH Flags,destination_Fwd Header Length,destination_Bwd Header Length,destination_Fwd Packets/s,destination_Bwd Packets/s,destination_Min Packet Length,destination_Max Packet Length,destination_Packet Length Mean,destination_Packet Length Std,destination_Packet Length Variance,destination_FIN Flag Count,destination_SYN Flag Count,destination_RST Flag Count,destination_PSH Flag Count,destination_ACK Flag Count,destination_URG Flag Count,destination_ECE Flag Count,destination_Down/Up Ratio,destination_Average Packet Size,destination_Avg Fwd Segment Size,destination_Avg Bwd Segment Size,destination_Fwd Header Length.1,destination_Subflow Fwd Packets,destination_Subflow Fwd Bytes,destination_Subflow Bwd Packets,destination_Subflow Bwd Bytes,destination_Init_Win_bytes_forward,destination_Init_Win_bytes_backward,destination_act_data_pkt_fwd,destination_min_seg_size_forward,destination_Active Mean,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min
Destination IP,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
101.201.103.96,-0.222908,-0.119495,-0.089164,-0.125195,-0.053023,-0.247379,-0.427194,-0.308106,-0.209924,-0.306943,-0.396604,-0.345473,-0.283668,-0.043514,-0.324603,0.026335,-0.157693,-0.152362,0.899732,-0.234724,-0.106533,-0.197619,-0.180948,-0.052982,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,-0.146348,-0.091747,-0.096237,-0.258783,-0.338138,-0.539775,-0.315617,-0.376670,-0.327712,-0.218177,-0.090300,-0.146348,-0.008568,0.715962,-0.375267,-0.216354,-0.008568,0.014486,-0.403217,-0.308106,-0.345473,-0.091747,-0.119495,-0.125195,-0.089164,-0.053023,1.264459,-0.172942,-0.116102,1.362622,-0.073218,-0.087551,-0.106105,-0.051450,-0.184749,-0.050323,-0.185562,-0.179561
101.37.178.168,1.275165,0.282866,0.214647,0.231581,0.058807,0.457160,-0.311741,0.248409,0.410524,1.199494,-0.507229,0.897790,1.425772,-0.041704,-0.284353,0.378161,0.376824,0.388250,-0.030152,1.097257,0.250231,0.840928,0.411927,-0.077640,1.327508,0.223301,0.938685,0.449826,-0.083687,-0.146348,0.187457,0.204900,-0.210503,-0.338176,-0.390666,1.074786,0.904852,1.175253,0.670498,-0.090300,-0.146348,-0.008568,-0.269957,1.043417,-0.216354,-0.008568,-0.852620,0.841221,0.248409,0.897790,0.187457,0.282866,0.231581,0.214647,0.058807,-0.466669,-0.171413,0.420562,-1.133138,0.469592,0.993973,1.057269,0.250750,0.412852,0.205113,0.414479,0.354411
103.203.138.103,1.276680,0.398087,0.131629,0.286223,0.018481,0.645236,-0.427194,0.525963,0.718018,2.214970,-0.507229,2.796398,3.636888,-0.043496,-0.324621,0.389409,0.394212,0.465527,-0.030315,1.293294,0.214149,0.791564,0.479914,-0.077669,1.127531,0.181160,0.395291,0.143560,-0.083591,-0.146348,0.287877,0.147077,-0.258793,-0.338169,-0.539775,2.004686,1.643587,2.514097,1.391073,-0.090300,-0.146348,-0.008568,0.715962,-0.375267,-0.216354,-0.008568,-1.888918,1.628516,0.525963,2.796398,0.287877,0.398087,0.286223,0.131629,0.018481,-0.207236,-0.095843,0.571499,-1.133138,0.622719,0.227488,0.674841,0.657206,0.251943,-0.046996,0.241313,0.261917
103.23.4.38,-0.224086,0.103632,-0.018510

In [12]:
destination_features = (
    graph_df
    .groupby("Destination IP")[traffic_features]
    .mean()
)

destination_features.columns = [
    f"destination_{column}"
    for column in destination_features.columns
]

destination_features.head()

,destination_Flow Duration,destination_Total Fwd Packets,destination_Total Backward Packets,destination_Total Length of Fwd Packets,destination_Total Length of Bwd Packets,destination_Fwd Packet Length Max,destination_Fwd Packet Length Min,destination_Fwd Packet Length Mean,destination_Fwd Packet Length Std,destination_Bwd Packet Length Max,destination_Bwd Packet Length Min,destination_Bwd Packet Length Mean,destination_Bwd Packet Length Std,destination_Flow Bytes/s,destination_Flow Packets/s,destination_Flow IAT Mean,destination_Flow IAT Std,destination_Flow IAT Max,destination_Flow IAT Min,destination_Fwd IAT Total,destination_Fwd IAT Mean,destination_Fwd IAT Std,destination_Fwd IAT Max,destination_Fwd IAT Min,destination_Bwd IAT Total,destination_Bwd IAT Mean,destination_Bwd IAT Std,destination_Bwd IAT Max,destination_Bwd IAT Min,destination_Fwd PSH Flags,destination_Fwd Header Length,destination_Bwd Header Length,destination_Fwd Packets/s,destination_Bwd Packets/s,destination_Min Packet Length,destination_Max Packet Length,destination_Packet Length Mean,destination_Packet Length Std,destination_Packet Length Variance,destination_FIN Flag Count,destination_SYN Flag Count,destination_RST Flag Count,destination_PSH Flag Count,destination_ACK Flag Count,destination_URG Flag Count,destination_ECE Flag Count,destination_Down/Up Ratio,destination_Average Packet Size,destination_Avg Fwd Segment Size,destination_Avg Bwd Segment Size,destination_Fwd Header Length.1,destination_Subflow Fwd Packets,destination_Subflow Fwd Bytes,destination_Subflow Bwd Packets,destination_Subflow Bwd Bytes,destination_Init_Win_bytes_forward,destination_Init_Win_bytes_backward,destination_act_data_pkt_fwd,destination_min_seg_size_forward,destination_Active Mean,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min
Destination IP,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
101.201.103.96,-0.222908,-0.119495,-0.089164,-0.125195,-0.053023,-0.247379,-0.427194,-0.308106,-0.209924,-0.306943,-0.396604,-0.345473,-0.283668,-0.043514,-0.324603,0.026335,-0.157693,-0.152362,0.899732,-0.234724,-0.106533,-0.197619,-0.180948,-0.052982,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,-0.146348,-0.091747,-0.096237,-0.258783,-0.338138,-0.539775,-0.315617,-0.376670,-0.327712,-0.218177,-0.090300,-0.146348,-0.008568,0.715962,-0.375267,-0.216354,-0.008568,0.014486,-0.403217,-0.308106,-0.345473,-0.091747,-0.119495,-0.125195,-0.089164,-0.053023,1.264459,-0.172942,-0.116102,1.362622,-0.073218,-0.087551,-0.106105,-0.051450,-0.184749,-0.050323,-0.185562,-0.179561
101.37.178.168,1.275165,0.282866,0.214647,0.231581,0.058807,0.457160,-0.311741,0.248409,0.410524,1.199494,-0.507229,0.897790,1.425772,-0.041704,-0.284353,0.378161,0.376824,0.388250,-0.030152,1.097257,0.250231,0.840928,0.411927,-0.077640,1.327508,0.223301,0.938685,0.449826,-0.083687,-0.146348,0.187457,0.204900,-0.210503,-0.338176,-0.390666,1.074786,0.904852,1.175253,0.670498,-0.090300,-0.146348,-0.008568,-0.269957,1.043417,-0.216354,-0.008568,-0.852620,0.841221,0.248409,0.897790,0.187457,0.282866,0.231581,0.214647,0.058807,-0.466669,-0.171413,0.420562,-1.133138,0.469592,0.993973,1.057269,0.250750,0.412852,0.205113,0.414479,0.354411
103.203.138.103,1.276680,0.398087,0.131629,0.286223,0.018481,0.645236,-0.427194,0.525963,0.718018,2.214970,-0.507229,2.796398,3.636888,-0.043496,-0.324621,0.389409,0.394212,0.465527,-0.030315,1.293294,0.214149,0.791564,0.479914,-0.077669,1.127531,0.181160,0.395291,0.143560,-0.083591,-0.146348,0.287877,0.147077,-0.258793,-0.338169,-0.539775,2.004686,1.643587,2.514097,1.391073,-0.090300,-0.146348,-0.008568,0.715962,-0.375267,-0.216354,-0.008568,-1.888918,1.628516,0.525963,2.796398,0.287877,0.398087,0.286223,0.131629,0.018481,-0.207236,-0.095843,0.571499,-1.133138,0.622719,0.227488,0.674841,0.657206,0.251943,-0.046996,0.241313,0.261917
103.23.4.38,-0.224086,0.103632,-0.018510

In [13]:
source_counts = (
    graph_df
    .groupby("Source IP")
    .size()
    .rename("source_flow_count")
)

source_counts.head()

Source IP
101.37.178.168     1
103.203.138.103    4
103.23.4.38        1
103.235.247.8      5
103.235.252.17     1
Name: source_flow_count, dtype: int64

In [14]:
destination_counts = (
    graph_df
    .groupby("Destination IP")
    .size()
    .rename("destination_flow_count")
)

destination_counts.head()

Destination IP
101.201.103.96      7
101.37.178.168     15
103.203.138.103     4
103.23.4.38         2
103.235.247.8       4
Name: destination_flow_count, dtype: int64

In [15]:
node_features = pd.concat(
    [
        source_features,
        destination_features,
        source_counts,
        destination_counts
    ],
    axis=1
)

node_features = node_features.fillna(0)

node_features.head()

,source_Flow Duration,source_Total Fwd Packets,source_Total Backward Packets,source_Total Length of Fwd Packets,source_Total Length of Bwd Packets,source_Fwd Packet Length Max,source_Fwd Packet Length Min,source_Fwd Packet Length Mean,source_Fwd Packet Length Std,source_Bwd Packet Length Max,source_Bwd Packet Length Min,source_Bwd Packet Length Mean,source_Bwd Packet Length Std,source_Flow Bytes/s,source_Flow Packets/s,source_Flow IAT Mean,source_Flow IAT Std,source_Flow IAT Max,source_Flow IAT Min,source_Fwd IAT Total,source_Fwd IAT Mean,source_Fwd IAT Std,source_Fwd IAT Max,source_Fwd IAT Min,source_Bwd IAT Total,source_Bwd IAT Mean,source_Bwd IAT Std,source_Bwd IAT Max,source_Bwd IAT Min,source_Fwd PSH Flags,source_Fwd Header Length,source_Bwd Header Length,source_Fwd Packets/s,source_Bwd Packets/s,source_Min Packet Length,source_Max Packet Length,source_Packet Length Mean,source_Packet Length Std,source_Packet Length Variance,source_FIN Flag Count,source_SYN Flag Count,source_RST Flag Count,source_PSH Flag Count,source_ACK Flag Count,source_URG Flag Count,source_ECE Flag Count,source_Down/Up Ratio,source_Average Packet Size,source_Avg Fwd Segment Size,source_Avg Bwd Segment Size,source_Fwd Header Length.1,source_Subflow Fwd Packets,source_Subflow Fwd Bytes,source_Subflow Bwd Packets,source_Subflow Bwd Bytes,source_Init_Win_bytes_forward,source_Init_Win_bytes_backward,source_act_data_pkt_fwd,source_min_seg_size_forward,source_Active Mean,source_Active Std,source_Active Max,source_Active Min,source_Idle Mean,source_Idle Std,source_Idle Max,source_Idle Min,destination_Flow Duration,destination_Total Fwd Packets,destination_Total Backward Packets,destination_Total Length of Fwd Packets,destination_Total Length of Bwd Packets,destination_Fwd Packet Length Max,destination_Fwd Packet Length Min,destination_Fwd Packet Length Mean,destination_Fwd Packet Length Std,destination_Bwd Packet Length Max,destination_Bwd Packet Length Min,destination_Bwd Packet Length Mean,destination_Bwd Packet Length Std,destination_Flow Bytes/s,destination_Flow Packets/s,destination_Flow IAT Mean,destination_Flow IAT Std,destination_Flow IAT Max,destination_Flow IAT Min,destination_Fwd IAT Total,destination_Fwd IAT Mean,destination_Fwd IAT Std,destination_Fwd IAT Max,destination_Fwd IAT Min,destination_Bwd IAT Total,destination_Bwd IAT Mean,destination_Bwd IAT Std,destination_Bwd IAT Max,destination_Bwd IAT Min,destination_Fwd PSH Flags,destination_Fwd Header Length,destination_Bwd Header Length,destination_Fwd Packets/s,destination_Bwd Packets/s,destination_Min Packet Length,destination_Max Packet Length,destination_Packet Length Mean,destination_Packet Length Std,destination_Packet Length Variance,destination_FIN Flag Count,destination_SYN Flag Count,destination_RST Flag Count,destination_PSH Flag Count,destination_ACK Flag Count,destination_URG Flag Count,destination_ECE Flag Count,destination_Down/Up Ratio,destination_Average Packet Size,destination_Avg Fwd Segment Size,destination_Avg Bwd Segment Size,destination_Fwd Header Length.1,destination_Subflow Fwd Packets,destination_Subflow Fwd Bytes,destination_Subflow Bwd Packets,destination_Subflow Bwd Bytes,destination_Init_Win_bytes_forward,destination_Init_Win_bytes_backward,destination_act_data_pkt_fwd,destination_min_seg_size_forward,destination_Active Mean,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min,source_flow_count,destination_flow_count
101.37.178.168,-0.233453,-0.075601,-0.089164,-0.118767,-0.053023,-0.229084,-0.179794,-0.231604,-0.209924,-0.306943,-0.396604,-0.345473,-0.283668,-0.043512,-0.324571,-0.099454,-0.125884,-0.172413,-0.030272,-0.229113,-0.086079,-0.197619,-0.167388,-0.031810,-0.233930,-0.127793,-0.167164,-0.180284,-0.083687,-0.146348,-0.103088,-0.096237,-0.258753,-0.338118,-0.220255,-0.315617,-0.343325,-0.346768,-0.218248,-0.0903,-0.146348,-0.008568,-1.396722,2.664771,4.622

In [16]:
node_features = node_features.reset_index()

node_features = node_features.rename(
    columns={"index": "ip_address"}
)

print("Node feature shape:", node_features.shape)

Node feature shape: (4404, 137)


C:\Users\daksh\AppData\Local\Temp\ipykernel_8400\713168015.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  node_features = node_features.reset_index()


In [17]:
if "Source IP" in node_features.columns:
    node_features = node_features.rename(
        columns={"Source IP": "ip_address"}
    )

if "Destination IP" in node_features.columns:
    node_features = node_features.rename(
        columns={"Destination IP": "ip_address"}
    )

print(node_features.columns.tolist())

['ip_address', 'source_Flow Duration', 'source_Total Fwd Packets', 'source_Total Backward Packets', 'source_Total Length of Fwd Packets', 'source_Total Length of Bwd Packets', 'source_Fwd Packet Length Max', 'source_Fwd Packet Length Min', 'source_Fwd Packet Length Mean', 'source_Fwd Packet Length Std', 'source_Bwd Packet Length Max', 'source_Bwd Packet Length Min', 'source_Bwd Packet Length Mean', 'source_Bwd Packet Length Std', 'source_Flow Bytes/s', 'source_Flow Packets/s', 'source_Flow IAT Mean', 'source_Flow IAT Std', 'source_Flow IAT Max', 'source_Flow IAT Min', 'source_Fwd IAT Total', 'source_Fwd IAT Mean', 'source_Fwd IAT Std', 'source_Fwd IAT Max', 'source_Fwd IAT Min', 'source_Bwd IAT Total', 'source_Bwd IAT Mean', 'source_Bwd IAT Std', 'source_Bwd IAT Max', 'source_Bwd IAT Min', 'source_Fwd PSH Flags', 'source_Fwd Header Length', 'source_Bwd Header Length', 'source_Fwd Packets/s', 'source_Bwd Packets/s', 'source_Min Packet Length', 'source_Max Packet Length', 'source_Packet 

In [18]:
source_attack_nodes = set(
    graph_df.loc[
        graph_df["Binary_Label"] == 1,
        "Source IP"
    ]
)

destination_attack_nodes = set(
    graph_df.loc[
        graph_df["Binary_Label"] == 1,
        "Destination IP"
    ]
)

attack_nodes = (
    source_attack_nodes |
    destination_attack_nodes
)

node_labels = pd.DataFrame({
    "ip_address": node_features["ip_address"],
    "node_label": node_features["ip_address"]
    .isin(attack_nodes)
    .astype(int)
})

node_labels.head()

,ip_address,node_label
0,101.37.178.168,0
1,103.203.138.103,0
2,103.23.4.38,0
3,103.235.247.8,0
4,103.235.252.17,0


In [19]:
print(
    node_labels["node_label"].value_counts()
)

node_label
0    4402
1       2
Name: count, dtype: int64


In [20]:
feature_columns = [
    column
    for column in node_features.columns
    if column != "ip_address"
]

scaler = StandardScaler()

node_features_scaled = scaler.fit_transform(
    node_features[feature_columns]
)

node_features_scaled = pd.DataFrame(
    node_features_scaled,
    columns=feature_columns
)

print("Node features standardized.")

Node features standardized.


In [21]:
node_features_final = pd.concat(
    [
        node_features[["ip_address"]].reset_index(drop=True),
        node_features_scaled.reset_index(drop=True)
    ],
    axis=1
)

node_features_final.head()

,ip_address,source_Flow Duration,source_Total Fwd Packets,source_Total Backward Packets,source_Total Length of Fwd Packets,source_Total Length of Bwd Packets,source_Fwd Packet Length Max,source_Fwd Packet Length Min,source_Fwd Packet Length Mean,source_Fwd Packet Length Std,source_Bwd Packet Length Max,source_Bwd Packet Length Min,source_Bwd Packet Length Mean,source_Bwd Packet Length Std,source_Flow Bytes/s,source_Flow Packets/s,source_Flow IAT Mean,source_Flow IAT Std,source_Flow IAT Max,source_Flow IAT Min,source_Fwd IAT Total,source_Fwd IAT Mean,source_Fwd IAT Std,source_Fwd IAT Max,source_Fwd IAT Min,source_Bwd IAT Total,source_Bwd IAT Mean,source_Bwd IAT Std,source_Bwd IAT Max,source_Bwd IAT Min,source_Fwd PSH Flags,source_Fwd Header Length,source_Bwd Header Length,source_Fwd Packets/s,source_Bwd Packets/s,source_Min Packet Length,source_Max Packet Length,source_Packet Length Mean,source_Packet Length Std,source_Packet Length Variance,source_FIN Flag Count,source_SYN Flag Count,source_RST Flag Count,source_PSH Flag Count,source_ACK Flag Count,source_URG Flag Count,source_ECE Flag Count,source_Down/Up Ratio,source_Average Packet Size,source_Avg Fwd Segment Size,source_Avg Bwd Segment Size,source_Fwd Header Length.1,source_Subflow Fwd Packets,source_Subflow Fwd Bytes,source_Subflow Bwd Packets,source_Subflow Bwd Bytes,source_Init_Win_bytes_forward,source_Init_Win_bytes_backward,source_act_data_pkt_fwd,source_min_seg_size_forward,source_Active Mean,source_Active Std,source_Active Max,source_Active Min,source_Idle Mean,source_Idle Std,source_Idle Max,source_Idle Min,destination_Flow Duration,destination_Total Fwd Packets,destination_Total Backward Packets,destination_Total Length of Fwd Packets,destination_Total Length of Bwd Packets,destination_Fwd Packet Length Max,destination_Fwd Packet Length Min,destination_Fwd Packet Length Mean,destination_Fwd Packet Length Std,destination_Bwd Packet Length Max,destination_Bwd Packet Length Min,destination_Bwd Packet Length Mean,destination_Bwd Packet Length Std,destination_Flow Bytes/s,destination_Flow Packets/s,destination_Flow IAT Mean,destination_Flow IAT Std,destination_Flow IAT Max,destination_Flow IAT Min,destination_Fwd IAT Total,destination_Fwd IAT Mean,destination_Fwd IAT Std,destination_Fwd IAT Max,destination_Fwd IAT Min,destination_Bwd IAT Total,destination_Bwd IAT Mean,destination_Bwd IAT Std,destination_Bwd IAT Max,destination_Bwd IAT Min,destination_Fwd PSH Flags,destination_Fwd Header Length,destination_Bwd Header Length,destination_Fwd Packets/s,destination_Bwd Packets/s,destination_Min Packet Length,destination_Max Packet Length,destination_Packet Length Mean,destination_Packet Length Std,destination_Packet Length Variance,destination_FIN Flag Count,destination_SYN Flag Count,destination_RST Flag Count,destination_PSH Flag Count,destination_ACK Flag Count,destination_URG Flag Count,destination_ECE Flag Count,destination_Down/Up Ratio,destination_Average Packet Size,destination_Avg Fwd Segment Size,destination_Avg Bwd Segment Size,destination_Fwd Header Length.1,destination_Subflow Fwd Packets,destination_Subflow Fwd Bytes,destination_Subflow Bwd Packets,destination_Subflow Bwd Bytes,destination_Init_Win_bytes_forward,destination_Init_Win_bytes_backward,destination_act_data_pkt_fwd,destination_min_seg_size_forward,destination_Active Mean,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min,source_flow_count,destination_flow_count
0,101.37.178.168,-0.263543,-0.032299,-0.110270,-0.067572,-0.422974,-0.306618,-0.019509,-0.190242,-0.347949,-0.388739,-0.084009,-0.321179,-0.432513,-0.316278,-0.704622,0.229281,0.226828,-0.101002,-0.072216,-0.258186,0.304249,-0.417828,-0.090046,0.653511,-0.441191,-0.440376,-0.436003,-0.442383,-0.418423,-0.513973,-0.306966,-0.235480,-0.645873,-0.521910,0.233676,-0.433612,-0.325556,-0.442121,-0.255532,-0.092894,-0.513973,-0.387939,-0.452298,0.45

In [22]:
final_nodes = node_features_final.merge(
    node_labels,
    on="ip_address",
    how="left"
)

final_nodes = final_nodes.reset_index(drop=True)

final_nodes.insert(
    0,
    "node_id",
    range(len(final_nodes))
)

print("Final node table shape:", final_nodes.shape)

final_nodes.head()

Final node table shape: (4404, 139)


,node_id,ip_address,source_Flow Duration,source_Total Fwd Packets,source_Total Backward Packets,source_Total Length of Fwd Packets,source_Total Length of Bwd Packets,source_Fwd Packet Length Max,source_Fwd Packet Length Min,source_Fwd Packet Length Mean,source_Fwd Packet Length Std,source_Bwd Packet Length Max,source_Bwd Packet Length Min,source_Bwd Packet Length Mean,source_Bwd Packet Length Std,source_Flow Bytes/s,source_Flow Packets/s,source_Flow IAT Mean,source_Flow IAT Std,source_Flow IAT Max,source_Flow IAT Min,source_Fwd IAT Total,source_Fwd IAT Mean,source_Fwd IAT Std,source_Fwd IAT Max,source_Fwd IAT Min,source_Bwd IAT Total,source_Bwd IAT Mean,source_Bwd IAT Std,source_Bwd IAT Max,source_Bwd IAT Min,source_Fwd PSH Flags,source_Fwd Header Length,source_Bwd Header Length,source_Fwd Packets/s,source_Bwd Packets/s,source_Min Packet Length,source_Max Packet Length,source_Packet Length Mean,source_Packet Length Std,source_Packet Length Variance,source_FIN Flag Count,source_SYN Flag Count,source_RST Flag Count,source_PSH Flag Count,source_ACK Flag Count,source_URG Flag Count,source_ECE Flag Count,source_Down/Up Ratio,source_Average Packet Size,source_Avg Fwd Segment Size,source_Avg Bwd Segment Size,source_Fwd Header Length.1,source_Subflow Fwd Packets,source_Subflow Fwd Bytes,source_Subflow Bwd Packets,source_Subflow Bwd Bytes,source_Init_Win_bytes_forward,source_Init_Win_bytes_backward,source_act_data_pkt_fwd,source_min_seg_size_forward,source_Active Mean,source_Active Std,source_Active Max,source_Active Min,source_Idle Mean,source_Idle Std,source_Idle Max,source_Idle Min,destination_Flow Duration,destination_Total Fwd Packets,destination_Total Backward Packets,destination_Total Length of Fwd Packets,destination_Total Length of Bwd Packets,destination_Fwd Packet Length Max,destination_Fwd Packet Length Min,destination_Fwd Packet Length Mean,destination_Fwd Packet Length Std,destination_Bwd Packet Length Max,destination_Bwd Packet Length Min,destination_Bwd Packet Length Mean,destination_Bwd Packet Length Std,destination_Flow Bytes/s,destination_Flow Packets/s,destination_Flow IAT Mean,destination_Flow IAT Std,destination_Flow IAT Max,destination_Flow IAT Min,destination_Fwd IAT Total,destination_Fwd IAT Mean,destination_Fwd IAT Std,destination_Fwd IAT Max,destination_Fwd IAT Min,destination_Bwd IAT Total,destination_Bwd IAT Mean,destination_Bwd IAT Std,destination_Bwd IAT Max,destination_Bwd IAT Min,destination_Fwd PSH Flags,destination_Fwd Header Length,destination_Bwd Header Length,destination_Fwd Packets/s,destination_Bwd Packets/s,destination_Min Packet Length,destination_Max Packet Length,destination_Packet Length Mean,destination_Packet Length Std,destination_Packet Length Variance,destination_FIN Flag Count,destination_SYN Flag Count,destination_RST Flag Count,destination_PSH Flag Count,destination_ACK Flag Count,destination_URG Flag Count,destination_ECE Flag Count,destination_Down/Up Ratio,destination_Average Packet Size,destination_Avg Fwd Segment Size,destination_Avg Bwd Segment Size,destination_Fwd Header Length.1,destination_Subflow Fwd Packets,destination_Subflow Fwd Bytes,destination_Subflow Bwd Packets,destination_Subflow Bwd Bytes,destination_Init_Win_bytes_forward,destination_Init_Win_bytes_backward,destination_act_data_pkt_fwd,destination_min_seg_size_forward,destination_Active Mean,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min,source_flow_count,destination_flow_count,node_label
0,0,101.37.178.168,-0.263543,-0.032299,-0.110270,-0.067572,-0.422974,-0.306618,-0.019509,-0.190242,-0.347949,-0.388739,-0.084009,-0.321179,-0.432513,-0.316278,-0.704622,0.229281,0.226828,-0.101002,-0.072216,-0.258186,0.304249,-0.417828,-0.090046,0.653511,-0.441191,-0.440376,-0.436003,-0.442383,-0.418423,-0.513973,-0.306966,-0.235480,-0.645873,-0.521910,0.233676,-0.433612,-0.325556,-0.442121,-0.255532,-0.092894,-0.513973,-0.

In [23]:
print(
    "Missing values:",
    final_nodes.isnull().sum().sum()
)

Missing values: 0


In [24]:
numeric_columns = final_nodes.select_dtypes(
    include=np.number
).columns

print(
    "Infinite values:",
    np.isinf(
        final_nodes[numeric_columns]
    ).sum().sum()
)

Infinite values: 0


In [25]:
print("Total nodes:", len(final_nodes))

print(
    "Normal nodes:",
    (final_nodes["node_label"] == 0).sum()
)

print(
    "Attack nodes:",
    (final_nodes["node_label"] == 1).sum()
)

print(
    "Feature count:",
    len(feature_columns)
)

Total nodes: 4404
Normal nodes: 4402
Attack nodes: 2
Feature count: 136


In [26]:
node_features_final.to_csv(
    OUTPUT_FEATURES,
    index=False
)

print(f"Saved: {OUTPUT_FEATURES}")

Saved: ../data/processed/node_features.csv


In [27]:
node_labels.to_csv(
    OUTPUT_LABELS,
    index=False
)

print(f"Saved: {OUTPUT_LABELS}")

Saved: ../data/processed/node_labels.csv


In [28]:
final_nodes.to_csv(
    OUTPUT_NODES,
    index=False
)

print(f"Saved: {OUTPUT_NODES}")

Saved: ../data/processed/final_nodes.csv


In [29]:
print("========== DAY 4 COMPLETE ==========")

print("Final nodes:", final_nodes.shape)

print("Node features:", node_features_final.shape)

print("Node labels:", node_labels.shape)

print("\nLabel distribution:")
print(node_labels["node_label"].value_counts())

print("\nSaved files:")
print("1. data/processed/node_features.csv")
print("2. data/processed/node_labels.csv")
print("3. data/processed/final_nodes.csv")

========== DAY 4 COMPLETE ==========
Final nodes: (4404, 139)
Node features: (4404, 137)
Node labels: (4404, 2)

Label distribution:
node_label
0    4402
1       2
Name: count, dtype: int64

Saved files:
1. data/processed/node_features.csv
2. data/processed/node_labels.csv
3. data/processed/final_nodes.csv
